# Fraud Detection System with Isolation Forest Algorithm Phase 2 (Based on Pyspark)

## A). Preparation

### 1.1). Importing Library

In [ ]:
import os
import sys
import logging
import threading
from pathlib import Path
from config import *
from pyspark.sql import SparkSession as SS

In [2]:
python_path = sys.executable
os.environ["PYSPARK_PYTHON"] = python_path
os.environ["PYSPARK_DRIVER_PYTHON"] = python_path

### 1.2.). Spark Session Start

In [3]:
import psutil
psutil.virtual_memory()

svmem(total=16366759936, available=3373793280, percent=79.4, used=12992966656, free=3373793280)

In [4]:
# Stopping the last Session Before start it

if SS.getActiveSession() is not None:
    SS.getActiveSession().stop()

#Dynamically Find CPU Cores
spark = (SS.builder \
    .appName("FraudDetection-IsolationForest-Phase2") \
    .master("local[6]") \
    .config("spark.pyspak.python", python_path) \
    .config("spark.pyspark.driver.python", python_path) \
    # SynapseML
    .config("spark.jars.packages", "com.microsoft.azure:synapseml_2.12:1.0.4") \
    .config("spark.jars.repositories", "https://mmlspark.azureedge.net/maven") \
    # Memory
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.maxResultSize", "2g") \
    # Performance
    .config("spark.sql.shuffle.partitions", "24") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.execution.arrow.spark.enabled", "true") \
    # Serializer
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    # Temp Dir
    .config("spark.local.dir", "C:/Users/umarl/AppData/Local/spark_temp")
    .getOrCreate())

spark.sparkContext.setLogLevel("ERROR")

spark

### 1.3). Logger

In [5]:
# Class Logger Manager
class LoggerManager:
    _loggers: dict= {}
    _lock = threading.Lock()
    def __init__(self, log_file: Path, name:str = "PySparkApp", level:int = logging.INFO):
        logger_key = f"{name}_{log_file}"
        with LoggerManager._lock:
            if logger_key in LoggerManager._loggers:
                self.logger = LoggerManager._loggers[logger_key]
                return
            
            logger = logging.getLogger(name)
            logger.setLevel(level)
            logger.propagate = False

            if not logger.handlers:
                # Console handler
                console_handler = logging.StreamHandler()
                console_handler.setLevel(level)
                console_handler.setFormatter(logging.Formatter(
                    "[%(levelname)s %(asctime)s - %(message)s]"
                ))
                logger.addHandler(console_handler)

                # file handler (driver-only, not applicable in executor)
                if log_file is not None:
                    log_path = Path(log_file)
                    log_path.parent.mkdir(parents=True, exist_ok=True)
                    file_handler = logging.FileHandler(log_path, mode='a')
                    file_handler.setLevel(level)
                    file_handler.setFormatter(logging.Formatter(
                        "%(asctime)s | %(levelname)s | %(name)s | %(message)s"
                    ))
                    logger.addHandler(file_handler)

            self.logger = logger
            LoggerManager._loggers[logger_key] = logger
    
    # Logging methods dengan stacklevel=2 agar lokasi caller tampil benar
    def info(self, message: str) -> None:
        self.logger.info(message, stacklevel=2)
    def error(self, message: str) -> None:
        self.logger.error(message, stacklevel=2)

# Di dalam UDF/mapPartitions - pakai logging standard langsung
def process_partition(rows):
    log = logging.getLogger("executor") # buat logger baru per-executor
    for row in rows:
        log.info(f"Processing: {row}")
        yield row


In [ ]:

log_dir_exp = LOG_DIR / f"logging_isolation_forest.txt"
logger = LoggerManager(log_file=log_dir_exp)

## B). Ingestion

### 2.1). Data Input

In [ ]:
# CSV to Parquet
import pandas as pd
excel_df = pd.read_excel(FOLDER_PATH / "realistic_transaction_dataset_38484.xlsx")
parquet_path = excel_df.to_parquet(FOLDER_PATH / "dummy_data_gpt.parquet", engine="pyarrow", coerce_timestamps="us", index=False)

In [10]:
df_path = "D:/Programming/DDVInternship/Data/Dataset/data sintesis/dummy_data_gpt.parquet"
raw_df = spark.read.parquet(df_path)
logger.info(f"Data Mentah Experimen IsolationForest: \n{raw_df.head(3)}")

[INFO 2026-06-26 13:46:48,583 - Data Mentah Experimen IsolationForest: 
[Row(TransactionDateTime=datetime.datetime(2024, 3, 23, 2, 41, 18), AccountCIFNo='CIF000001', TransactionAmount=640717), Row(TransactionDateTime=datetime.datetime(2024, 5, 22, 10, 56, 52), AccountCIFNo='CIF000001', TransactionAmount=762295), Row(TransactionDateTime=datetime.datetime(2024, 7, 25, 10, 13, 37), AccountCIFNo='CIF000001', TransactionAmount=868645)]]


### 2.2). Data Information

In [11]:
## showing the count, mean, stddev, min, and max.
desc_raw = raw_df.describe().dtypes
logger.info(f"Berikut Deskripsi Dataset Mentah: \n{desc_raw}")


[INFO 2026-06-26 13:46:58,699 - Berikut Deskripsi Dataset Mentah: 
[('summary', 'string'), ('AccountCIFNo', 'string'), ('TransactionAmount', 'string')]]


## C). Feature Engineering

### Wrapper

In [12]:
import builtins
from pyspark.sql.functions import sum, when, col
from pyspark.sql import DataFrame

## Cleaning Data Function
def cleaning_data (df:DataFrame, dropNull:bool = False, dropDup:bool = False, subset_null:list = None, subset_dup:list = None) -> tuple[dict, DataFrame]:
    """
    Params:
    dropNull = drop rows containing null values,
    dropDup = Drop duplicates rows,
    subset_null = Columns to consider for dropna (None = all columns),
    subset_dup = Columns to consider for dropDup (None = all columns)
    """
    
    if df is None or len(df.columns) == 0:
        return {}, df
    
    ## 1. Get Null counts on ORIGINAL DataFrame
    null_counts = df.select([sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns])
    null_dict = null_counts.first().asDict()

    total_rows = df.count()
    null_sum = {column: {
            "null_count": count,
            "null_percentage": builtins.round(count / total_rows * 100, 4) if total_rows > 0 else 0.0} for column, count in null_dict.items()}

    ## 2. Data Cleaning
    ### 2a. Drop duplicates
    if dropDup:
        df = df.dropDuplicates(subset=subset_dup)
    ### 2b. Drop Null
    if dropNull:
        df = df.dropna(subset=subset_null)

    ## Optional: add info about how many rows were removed
    rows_removed = total_rows - df.count()
    
    return null_sum, df, rows_removed

In [13]:
from pyspark.sql import functions as F
from pyspark.sql import Window

def feature_engineering(df:DataFrame):
    # sorting the datetime
    ## Feature 1 - hour_in_dec
    df = df.withColumn("timestamp", F.to_timestamp("TransactionDateTime"))
    df = df.withColumn("hour", F.hour("timestamp")) \
    .withColumn("minute", F.minute("timestamp")) \
    .withColumn("second", F.second("timestamp")) \
    .withColumn("day", F.dayofmonth("timestamp")) \
    .withColumn("weekday", F.weekday("timestamp")) \
    .withColumn("week", F.weekday("timestamp")) \
    .withColumn("month", F.month("timestamp"))

    df = df.withColumn(
    "hour_in_dec", F.col("hour") + F.col("minute")/60 + F.col("second")/3600
    )

    ## Feature 2 time_since_last_transaction
    ###  Using lag() Window Function
    window_acc_time = Window.partitionBy("AccountCIFNo").orderBy("timestamp")

    df = df.withColumn(
    "prev_timestamp", F.lag("timestamp").over(window_acc_time)
    )
    df = df.withColumn(
    "time_last_transaction", F.when(F.col("prev_timestamp").isNull(), 0) \
    .otherwise((F.unix_timestamp("timestamp") - F.unix_timestamp("prev_timestamp")) / 3600)).drop("prev_timestamp")

    ## Feature 3 -- Amount Transaction in Hourly
    window_acct_hour = Window.partitionBy("AccountCIFNo", "hour")
    df = df.withColumn("amt_transaction_hourly", F.sum("TransactionAmount").over(window_acct_hour) / 100_000)

    ## Feature 4 --- Amount Cumulative
    ### Using window sum with rangeBetween
    window_sum = Window.partitionBy("AccountCIFNo").orderBy("timestamp") \
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
    df = df.withColumn("amt_cumulative", F.sum("TransactionAmount").over(window_sum) / 100_000)

    ## Feature 5 --- Amount Diffrence from Last Transaction
    df = df.withColumn(
    "prev_amount", F.lag("TransactionAmount").over(window_acc_time)
    )
    df = df.withColumn(
    "amt_diff_last_transaction", F.when(F.col("prev_amount").isNull(), 0).otherwise(
        F.abs(F.col("TransactionAmount") - F.col("prev_amount")) / 100_000
    )
    ).drop("prev_amount")

    ## Feature 6 --- Amount Relative Transaction
    window_acct = Window.partitionBy("AccountCIFNo")
    df = df.withColumn(
    "avg_amount", F.avg("TransactionAmount").over(window_acct)
    )
    df = df.withColumn(
    "amt_relative_transaction", F.col("TransactionAmount") / F.col("avg_amount")
    ).drop("avg_amount")

    ## Feature 7 --- Transaction Frequency Daily
    window_acct_day = Window.partitionBy("AccountCIFNo").orderBy("day")
    df = df.withColumn("transaction_freq_daily", F.count("*").over(window_acct_day))

    ## Feature 8 --- Transaction Frequency Variability
    ### Hitung Count per AccountCIFNo per hour
    hourly_counts = df.groupBy("AccountCIFNo", "hour") \
    .agg(F.count("*").alias("hourly_count"))

    ### Hitung STD dari hourly_count per AccountCIFNo
    variability = hourly_counts.groupBy("AccountCIFNo") \
    .agg(F.stddev("hourly_count").alias("transaction_freq_variability"))

    ### Fillna(1.0) untuk nasabah yang hanya punya 1 jam transaksi
    variability = variability.fillna(1.0, subset=["transaction_freq_variability"])
    df = df.join(variability, on="AccountCIFNo", how="left")
    
    ## Drop Kolom-kolom yang tidak diperlukan
    drop_cols = ['AccountCIFNo', 'TransactionDateTime', 'TransactionAmount', 'timestamp', 'hour', 'minute', 'second', 'day', 'weekday', 'week', 'month']
    fix_df = df.drop(*drop_cols)
    
    return fix_df

### 3.1). Drop the Features to Features Selected

In [ ]:
# # execute the feature engineering function
# cols_to_drop = raw_df[
#   'Id', 'AccountID', 'AccountName', 'AccountType', 'AccountStatus',
#   'AccountCurrency', 'TransactionID', 'TransactionType', 'TransactionSOFType',
#   'TransactionStatus', 'ProductName', 'NoPelanggan', 'AccountNIK', 'timegetdata'
# ]
# df = raw_df.drop(*cols_to_drop)
# logger.info(f"Berikut Atribute data dan value yang akan digunakan: \n{df}")


### 3.2). Data Cleaning

In [14]:
## Drop Null and Duplicates
dict_all, df, after_removed_all = cleaning_data(raw_df, dropNull=True, dropDup=True)
total_before = raw_df.count()
total_after = df.count()
logger.info(f"The rows before cleaning data: {total_before}")
logger.info(f"Dictionary ALL raw Data: \n{dict_all}")
logger.info(f"All rows after cleaning (from null & duplicates): {total_after}")
logger.info(f"Total Removed Data: {after_removed_all}")

[INFO 2026-06-26 13:47:14,128 - The rows before cleaning data: 38484]
[INFO 2026-06-26 13:47:14,129 - Dictionary ALL raw Data: 
{'TransactionDateTime': {'null_count': 0, 'null_percentage': 0.0}, 'AccountCIFNo': {'null_count': 0, 'null_percentage': 0.0}, 'TransactionAmount': {'null_count': 0, 'null_percentage': 0.0}}]
[INFO 2026-06-26 13:47:14,130 - All rows after cleaning (from null & duplicates): 38484]
[INFO 2026-06-26 13:47:14,131 - Total Removed Data: 0]


### 3.3). Feature Engineering

In [15]:
fix_df = feature_engineering(df)
fix_df.show(5)
fix_df.count()

+------------------+---------------------+----------------------+--------------+-------------------------+------------------------+----------------------+----------------------------+
|       hour_in_dec|time_last_transaction|amt_transaction_hourly|amt_cumulative|amt_diff_last_transaction|amt_relative_transaction|transaction_freq_daily|transaction_freq_variability|
+------------------+---------------------+----------------------+--------------+-------------------------+------------------------+----------------------+----------------------------+
|2.6883333333333335|                  0.0|               6.40717|       6.40717|                      0.0|      0.7722962629032899|                     9|          0.7440238091428449|
| 9.579166666666666|    6982.913888888889|               9.17516|      40.45377|                  1.05367|      0.3997705360627315|                     4|          0.7440238091428449|
|            9.0175|   1099.6866666666667|               9.17516|     107.85137|

38484

## D). Build Pipeline

### Pipeline (Vector Assembler, Standardization, and IF Model)

In [17]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from synapse.ml.isolationforest import IsolationForest

def build_pipeline(feature_cols: list, if_config:dict) -> Pipeline:
    assembler = VectorAssembler(inputCols=feature_cols, outputCol="features", handleInvalid="skip")
    scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures", withMean=True, withStd=True)
    IFModel = (IsolationForest().setBootstrap(False).setFeaturesCol("scaledFeatures")
               .setPredictionCol("predictedLabel").setScoreCol("anomalyScore")
               .setNumEstimators(if_config["num_estimators"]).setMaxFeatures(if_config["max_features"])
               .setMaxSamples(if_config["max_samples"]).setContamination(if_config["contamination"])
               .setRandomSeed(if_config["random_seed"]))
    return Pipeline(stages=[assembler, scaler, IFModel])

def run_pipeline(df, pipeline: Pipeline):
    df = df.localCheckpoint(eager=True)
    model = pipeline.fit(df)
    preds = model.transform(df).localCheckpoint(eager=True)
    return model, preds

### Running the Pipeline

In [ ]:
pipeline = build_pipeline(FEATURE_COLS, IF_CONFIG)
model, preds = run_pipeline(fix_df, pipeline)

## E). Results

### Wrapper

In [19]:
from pyspark.ml.functions import vector_to_array
from pyspark.sql.functions import *

def fraction_results(results): # for pipeline_IF in drpipe
    # model fraction to state the isolation forest model fitted
    ifModel = model.stages[-1]
    # results after fitted
    preds = results.localCheckpoint(eager=True)
    scaled_df = preds.select("scaledFeatures")
    norm_df = (preds.filter(col(LABEL_COL) == 0).orderBy(asc(SCORE_COL)).drop("features", "scaledFeatures"))
    anom_df = (preds.filter(col(LABEL_COL) == 1).orderBy(desc(SCORE_COL)).drop("features", "scaledFeatures"))
    # make a std_fix_df (standardized fix_df)
    array_df = preds.withColumn("scaledArray", vector_to_array(col(FEAT_COL_STD)))
    std_cols = []
    for i, feat in enumerate(FEATURE_COLS):
        std_col = f"{feat}"
        feat_arr_df = array_df.withColumn(std_col, col("scaledArray")[i])
        std_cols.append(std_col)
        
    std_fix_df = feat_arr_df.select(*std_cols)

    return std_fix_df, norm_df, anom_df, scaled_df, ifModel


In [20]:
def results_summary(results:DataFrame) -> dict:
    """Statistik Deskriptif: distribusi label & skor per kelas."""
    preds = results
    total_count = preds.count()

    summ_dict = (
        preds.groupBy(LABEL_COL).agg(
            count("*").alias("count"),
            round((count("*") / total_count) * 100, 5).alias("percentage"),
            min(SCORE_COL).alias("min_score"),
            max(SCORE_COL).alias("max_score"),
            mean(SCORE_COL).alias("mean_score"),
            stddev(SCORE_COL).alias("stddev_score")).orderBy(LABEL_COL))
    summ_dict.show(truncate=False)

    ## Score Gap from each minimum score labels
    normal_stats = preds.filter(col(LABEL_COL) == 0).select(
        count("*").alias("n_normal"), max(SCORE_COL).alias("max_score"),).collect()[0]
    anomaly_stats = preds.filter(col(LABEL_COL) == 1).select(
        count("*").alias("n_anomalies"), min(SCORE_COL).alias("min_score"),).collect()[0]
    score_gap_min = float(anomaly_stats["min_score"] - normal_stats["max_score"])

    ## Score Gap from Quantiles Q1 Anomaly and Q99 Normal Scores
    norm_scores = preds.filter(col(LABEL_COL) == 0)
    anom_scores = preds.filter(col(LABEL_COL) == 1)
    anomP1 = anom_scores.stat.approxQuantile(SCORE_COL, [0.01], 0.05)[0]
    normP99 = norm_scores.stat.approxQuantile(SCORE_COL, [0.99], 0.05)[0]
    score_gap_quantile = (anomP1 - normP99)
    recom_thre = (anomP1+normP99) / 2 # midpoint
    print(f"Score Gap from Quantiles:       {score_gap_quantile:.8f}")
    print(f"Score Gap from Mininum Scores:  {score_gap_min:.8f}")
    print(f"Recommended Threshold: {recom_thre:.8f}")
    ## Overlap Check
    if anomP1 > normP99:
        print("Data is separated perfectly, nothing's overlap!")
    scores_dict = {"anomalies_scores":anom_scores, "normal_scores":norm_scores,
                "scores_gap_quantile": score_gap_quantile, "recom_thre":recom_thre}
    
    return scores_dict, summ_dict

def top_last_scores(preds:DataFrame)-> DataFrame:
    top = (
        preds.filter(col(LABEL_COL) == 1).orderBy(desc(SCORE_COL))
        .limit(20).select(LABEL_COL, SCORE_COL, *FEATURE_COLS))
    last = (
        preds.filter(col(LABEL_COL) == 1).orderBy(asc(SCORE_COL))
        .limit(20).select(LABEL_COL, SCORE_COL, *FEATURE_COLS))
    
    last_norm = (
        preds.filter(col(LABEL_COL) == 0).orderBy(desc(SCORE_COL))
        .limit(20).select(LABEL_COL, SCORE_COL, *FEATURE_COLS)
    )
    top_norm = (
        preds.filter(col(LABEL_COL) == 0).orderBy(asc(SCORE_COL))
        .limit(20).select(LABEL_COL, SCORE_COL, *FEATURE_COLS)
    )
    print("Berikut TOP 20 Data Anomali:")
    top.show(truncate=False)
    print("Berikut LAST 20 Data Anomali:")
    last.show(truncate=False)
    print("Berikut LAST 20 Data Normal:")
    last_norm.show(truncate=False)
    print("Berikut TOP 20 Data Normal:")
    top_norm.show(truncate=False)


### 4.1). Summarization

In [21]:
std_fix_df, norm_df, anom_df, scaled_df, ifModel = fraction_results(preds)

In [22]:
import pyspark.sql.functions as sqlf
cols_with_unit_100k = ['amt_transaction_hourly','amt_cumulative', 'amt_diff_last_transaction', 'amt_relative_transaction']
top_outlier_df = (anom_df.orderBy(desc(col("anomalyScore"))).drop("anomalyScore", "predictedLabel"))
top_outlier_final = top_outlier_df.select(
    *[
        (sqlf.col(c)*100_000).alias(c) if c in cols_with_unit_100k
        else sqlf.col(c)
        for c in top_outlier_df.columns])


top_outlier_final.show(truncate=False)

+------------------+---------------------+----------------------+--------------------+-------------------------+------------------------+----------------------+----------------------------+
|hour_in_dec       |time_last_transaction|amt_transaction_hourly|amt_cumulative      |amt_diff_last_transaction|amt_relative_transaction|transaction_freq_daily|transaction_freq_variability|
+------------------+---------------------+----------------------+--------------------+-------------------------+------------------------+----------------------+----------------------------+
|12.9              |592.7805555555556    |5.98727975E8          |2.678928279E9       |4.78575151E8             |1437141.8713147265      |72                    |3.481926122681169           |
|8.873888888888889 |90.005               |5.24408584E8          |3.8990950320000005E9|4.9834515199999994E8     |1141290.4695778647      |85                    |3.133729637015372           |
|12.83361111111111 |356.8375             |1.001379

In [ ]:
top_outlier_final.write.mode('overwrite').csv(OUTPUT_DIR / "outlier_data.csv")

## G. Spark Session Stop

In [23]:
import gc
try:
    spark.catalog.clearCache()
    try:
        spark.sharedState.cacheManager.clearCache()
    except:
        pass

    spark.stop()
    SS.clearActiveSession()
    SS.clearDefaultSession()
except Exception as e:
    print(f"Cleaning Warning: {e}")
finally:
    gc.collect()
    print("Spark Cleanup Completed!")

Cleaning Warning: type object 'SparkSession' has no attribute 'clearActiveSession'
Spark Cleanup Completed!
